In [ ]:
import Agama

In [ ]:
using LilGuys

In [ ]:
using CairoMakie

In [ ]:
import CSV
using DataFrames: DataFrame

In [ ]:
CairoMakie.activate!(type=:png)

In [ ]:
units = Agama.VASILIEV_UNITS

In [ ]:
pot_dir = "potentials/vasiliev24/L2M10"

In [ ]:
pot = Agama.Potential(file="$pot_dir/potential.ini")

In [ ]:
pot_lmc_init = Agama.Potential(file="$pot_dir/potential_lmc_init.ini")

In [ ]:

log_r = LinRange(-2, 2, 100)
r = exp10.(log_r)

fig = Figure()
ax = Axis(fig[1,1],
    xlabel = "radius / kpc",
    ylabel = "circular velocity / km/s"
    )
plot!(r, Agama.circular_velocity(pot_lmc_init, r, units) * V2KMS)
fig

In [ ]:

r = LinRange(0, 10, 100)

fig = Figure()
ax = Axis(fig[1,1],
    xlabel = "radius / kpc",
    ylabel = "potential",
    )
plot!(r, Agama.potential(pot_lmc_init, r' .* [1, 0, 0], units))
fig

In [ ]:

Φ_min = Agama.potential(pot_lmc_init, zeros(3), units)

In [ ]:
Φ_lmc(r) = Agama.potential(pot_lmc_init, [1,0,0] * r, units)
M_lmc(r) = Agama.enclosed_mass(pot_lmc_init, r, units)

In [ ]:
v_circ(r) = sqrt(M_lmc(r) / r)

In [ ]:
function r_circ(ϵ)
    result = NaN
    if nextfloat(Φ_lmc(0), 5) <= ϵ
        result = LilGuys.find_zero(
        r -> Φ_lmc(r) + 1/2 * M_lmc(r) / r - ϵ,
        (1e-10, 10000.))
    end
    return result
end

In [ ]:
Φ_lmc(0)

In [ ]:
function L_max(ϵ)
    r = r_circ(ϵ)
    v = v_circ(r)

    return r*v
end

In [ ]:
ϵ = LinRange(Φ_min, -0.05, 200)
lines(ϵ, L_max.(ϵ))

In [ ]:
action_mapper = Agama.ActionMapper(pot_lmc_init)
action_finder = Agama.ActionFinder(pot_lmc_init)

In [ ]:
function sample_coordinate(ϵ, L)
    ϕ = rand() * 2π # orbital phase
    L_vec = LilGuys.rand_unit() * L
    θ = rand() * 2π # direction of pericentre in orbital plane
    

    r0 = r_circ(ϵ)
    v0 = sqrt(2*(ϵ - Φ_lmc(r0)))
    x_hat = LilGuys.rand_unit()[:, 1]
    x = r0 * x_hat

    v_tan = L / r0
    v_rad = sqrt(v0^2 - v_tan^2)
    vec_tan = LilGuys.angular_momenta(LilGuys.rand_unit()[:, 1], x)
    vec_tan /= radii(vec_tan)

    
    v = v_tan * vec_tan + v_rad * x_hat

    actions = Agama.actions(action_finder, x, v, units)
    return Agama.from_actions(action_mapper, actions, rand(3) * 2π, units)
end
    

In [ ]:
using Arya

In [ ]:
LilGuys.rand_unit()

In [ ]:
function sample_relative_coordinate(e, l)
    ϵ = e * Φ_min
    L = L_max(ϵ) * l

    return sample_coordinate(ϵ, L)
end

In [ ]:
pos, vel = sample_relative_coordinate(0.8, 1)

In [ ]:
radii(LilGuys.angular_momenta(pos, vel))

In [ ]:
Φ_lmc(radii(pos)) + 1/2 * radii(vel)^2

In [ ]:
function sample_coordinates(N)
    positions = Matrix{Float64}(undef, 3, N)
    velocities = Matrix{Float64}(undef, 3, N)


    for i in 1:N
        e = 0.01 + 0.98 * rand()
        l = rand()
        pos, vel = sample_relative_coordinate(e, l)
        positions[:, i] = pos
        velocities[:, i] = vel
    end

    return positions, velocities
end

# Sampling by profile

In [ ]:
prof_halo = Agama.Potential(type="Spheroid", densityNorm=1, alpha=1e-10, beta=1, gamma=1, 
        scaleRadius=1, outerCutoffRadius=100, cutoffStrength=3)
        

In [ ]:
df = Agama.DistributionFunction(type="quasiSpherical", potential=pot_lmc_init._py, density=prof_halo._py)

gm = Agama.GalaxyModel(pot_lmc_init, df)

In [ ]:
positions, velocities = Agama.sample(gm, 1000, units)

In [ ]:
positions[2, :]

In [ ]:
hist(log10.(radii(positions)))

In [ ]:
scatter(positions[1, :], positions[2, :], markersize=1, alpha=0.3, color=:black)

In [ ]:
function get_lmc_orbit(input)
    lmc_file = joinpath(input, "trajlmc.txt")
    df_lmc = lmc_traj = CSV.read(lmc_file, DataFrame, delim=" ", header = [:time, :x, :y, :z, :v_x, :v_y, :v_z], ignorerepeated=true, ntasks=1)

    pos = hcat(df_lmc.x, df_lmc.y, df_lmc.z)'
    vel = hcat(df_lmc.v_x, df_lmc.v_y, df_lmc.v_z)'

    # convert to code units
    t = df_lmc.time .* Agama.time_scale(Agama.VASILIEV_UNITS) 
    vel .*= Agama.velocity_scale(Agama.VASILIEV_UNITS) 

    orbit_lmc = Orbit(times=t, positions=pos, velocities=vel)
end


In [ ]:
lmc_orbit = get_lmc_orbit(pot_dir)

In [ ]:
time0 = lmc_orbit.times[1]

In [ ]:
pos0 = lmc_orbit.positions[:, 1]
vel0 = lmc_orbit.velocities[:, 1]

In [ ]:
ϵs = radii(velocities) .^2/2 .+ Φ_lmc.(radii(positions))

In [ ]:
hist(ϵs)

In [ ]:
time0 * T2GYR

In [ ]:
orbits = Agama.orbit(pot, positions .+ pos0, velocities .+ vel0, units,
    timerange=(time0, 0)
    );

In [ ]:
fill((-300, 300), 3)

In [ ]:
plot_rmax = 300
plot_limits = ((-plot_rmax, plot_rmax), (-plot_rmax, plot_rmax), (-plot_rmax, plot_rmax), )

In [ ]:
LilGuys.plot_xyz(Agama.positions.(orbits)..., limits=plot_limits, alpha=0.03, 
color=:black)



In [ ]:
pos_f = hcat([Agama.positions(o)[:, end] for o in orbits]...)
vel_f = hcat([Agama.velocities(o)[:, end] for o in orbits]...)


In [ ]:
fig = LilGuys.plot_xyz(pos_f, plot=:scatter, limits=plot_limits, markersize=3, alpha=0.2, color=:black)
LilGuys.plot_xyz!(fig.content, lmc_orbit.positions)

fig

In [ ]:
?eachindex

In [ ]:
gc_lmc = [LilGuys.Galactocentric(lmc_orbit.positions[:, i], lmc_orbit.velocities[:, i]/ V2KMS) for i in 1:length(lmc_orbit)]
icrs_lmc = LilGuys.transform.(ICRS, gc_lmc)

In [ ]:
gc = [LilGuys.Galactocentric(pos_f[:, i], vel_f[:, i] / V2KMS) for i in 1:size(pos_f, 2)];
icrs = LilGuys.transform.(ICRS, gc)

In [ ]:
using GeoMakie

In [ ]:
fig = Figure()
ax = GeoAxis(fig[1,1];
    dest = "+proj=hammer",
    limits = (0., 360, -90, 90),
    yticklabelsvisible=false,
    xticklabelsvisible=false,
    yticklabelsize=8,
    xgridwidth=0.5,
    ygridwidth=0.5,
    valign=:center,
             
)
xlims!(-180, 180)

scatter!([c.ra for c in icrs_lmc], [c.dec for c in icrs_lmc], markersize =s, marker=:circle)

scatter!([c.ra for c in icrs], [c.dec for c in icrs], markersize=3, marker=:circle)
s = (lmc_orbit.times .- time0)/abs(time0) * 5


fig

In [ ]:
wrap_ra(ra) = @. ifelse(ra < 180, ra+360, ra)

In [ ]:
?ticks

In [ ]:
?modf

In [ ]:
fig = Figure()
ax = Axis(fig[1,1],
    xticks =[180, 270, 360, 450, 540], 
    xtickformat = x->string.(round.(Int, x .% 360)),
    xminorticks = 180:15:540,
    xlabel="ra",
    ylabel="dec"
)

scatter!(wrap_ra([c.ra for c in icrs_lmc]), [c.dec for c in icrs_lmc], markersize =s, marker=:circle)

scatter!(wrap_ra([c.ra for c in icrs]), [c.dec for c in icrs], markersize=3, marker=:circle)
arrows2d!(wrap_ra([c.ra for c in icrs])
    , [c.dec for c in icrs],
    [c.pmra for c in icrs], [c.pmdec for c in icrs],
    lengthscale=3, minshaftlength=0,
    tiplength=3/2, tipwidth=3,
    shaftwidth=1,
    )


s = (lmc_orbit.times .- time0)/abs(time0) * 5

xlims!(360+180, 180)


fig